# K-Means Clustering

## Co je K-Means Clustering?

K-Means je jeden z nejjednodušších a nejpoužívanějších shlukovacích (clusteringových) algoritmů. Patří do kategorie "učení bez učitele" (unsupervised learning), což znamená, že nepotřebuje označená data pro trénování.

Algoritmus rozděluje datové body do K předem definovaných nepřekrývajících se shluků (clusterů), kde každý datový bod patří do shluku s nejbližším středem (centroidem).

### Princip algoritmu K-Means:

1. **Inicializace** - Vybere se K náhodných bodů jako počáteční centroidy.
2. **Přiřazení** - Každý datový bod je přiřazen k nejbližšímu centroidu.
3. **Aktualizace** - Centroidy se přepočítají jako průměr všech bodů v daném shluku.
4. **Opakování** - Kroky 2 a 3 se opakují, dokud nedojde ke konvergenci (centroidy se už významně nemění).

### Matematické vyjádření:

K-Means se snaží minimalizovat součet čtverců vzdáleností mezi datovými body a centroidem jejich shluku:

$$J = \sum_{i=1}^{k} \sum_{x \in S_i} ||x - \mu_i||^2$$

kde:
- $J$ je funkce, kterou minimalizujeme
- $k$ je počet shluků
- $S_i$ je $i$-tý shluk
- $\mu_i$ je centroid $i$-tého shluku
- $x$ je datový bod ve shluku $S_i$

### Výhody a nevýhody K-Means:

**Výhody:**
- Jednoduchá implementace
- Škálovatelnost na velké datové sady
- Rychlá konvergence
- Garantované ukončení algoritmu

**Nevýhody:**
- Nutnost předem definovat počet shluků K
- Citlivost na volbu počátečních centroidů
- Předpokládá sférické (kulovité) shluky stejné velikosti
- Citlivost na odlehlé hodnoty
- Může uvíznout v lokálním minimu

### Kdy použít K-Means:
- Když hledáte sférické shluky v datech
- Když máte velké datové sady a potřebujete rychlý algoritmus
- Když předem znáte nebo máte dobrý odhad počtu shluků
- Pro předzpracování dat nebo snížení dimenzionality

In [ ]:
# Import potřebných knihoven
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.datasets import make_blobs, make_moons, load_iris, fetch_olivetti_faces
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D
from sklearn.metrics.pairwise import euclidean_distances
import time

# Nastavení vizuálního stylu
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

# Pro reprodukovatelnost výsledků
np.random.seed(42)

## 1. Jednoduchý příklad K-Means na syntetických datech

Nejprve vytvoříme a vizualizujeme jednoduchá syntetická data s několika zřetelně oddělenými shluky, abychom demonstrovali základní použití K-Means.

In [ ]:
# Vytvoření syntetických dat se 3 shluky
X, y = make_blobs(n_samples=300, centers=3, cluster_std=0.60, random_state=0)

# Vizualizace dat
plt.figure(figsize=(10, 6))
plt.scatter(X[:, 0], X[:, 1], s=50)
plt.title('Syntetická data se 3 shluky', fontsize=14)
plt.xlabel('První příznak', fontsize=12)
plt.ylabel('Druhý příznak', fontsize=12)
plt.colorbar()
plt.grid(True)
plt.show()

In [ ]:
# Aplikace K-Means s K=3
kmeans = KMeans(n_clusters=3, random_state=0)
cluster_labels = kmeans.fit_predict(X)
centroids = kmeans.cluster_centers_

# Vizualizace výsledků
plt.figure(figsize=(10, 6))
plt.scatter(X[:, 0], X[:, 1], c=cluster_labels, cmap='viridis', s=50)
plt.scatter(centroids[:, 0], centroids[:, 1], c='red', s=200, alpha=0.75, marker='X')
plt.title('Výsledky K-Means (K=3)', fontsize=14)
plt.xlabel('První příznak', fontsize=12)
plt.ylabel('Druhý příznak', fontsize=12)
plt.colorbar(label='Cluster ID')
plt.grid(True)
plt.show()

# Základní informace o výsledcích
print(f"Počet iterací algoritmu: {kmeans.n_iter_}")
print(f"Hodnota objektivní funkce (inertia): {kmeans.inertia_:.2f}")
print(f"Centroidy shluků:")
for i, centroid in enumerate(centroids):
    print(f"  Shluk {i}: {centroid}")
    
# Výpočet metrik kvality shlukování
silhouette = silhouette_score(X, cluster_labels)
ch_score = calinski_harabasz_score(X, cluster_labels)
db_score = davies_bouldin_score(X, cluster_labels)

print("\nMetriky kvality shlukování:")
print(f"  Silhouette koeficient: {silhouette:.3f} (vyšší je lepší, max=1)")
print(f"  Calinski-Harabasz index: {ch_score:.3f} (vyšší je lepší)")
print(f"  Davies-Bouldin index: {db_score:.3f} (nižší je lepší)")

### Vizualizace procesu K-Means

Pro lepší pochopení algoritmu si ukážeme, jak K-Means postupně konverguje k finálnímu řešení.

In [ ]:
def kmeans_step_by_step(X, n_clusters=3, max_iters=8):
    # Inicializace K-Means
    kmeans = KMeans(n_clusters=n_clusters, random_state=0, max_iter=1, n_init=1)
    
    # První inicializace centroidů
    kmeans.fit(X)
    centroids = kmeans.cluster_centers_
    labels = kmeans.labels_
    
    # Vytvoření mřížky pro vizualizaci
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    axes = axes.flatten()
    
    # Vizualizace počáteční inicializace
    axes[0].scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=50, alpha=0.7)
    axes[0].scatter(centroids[:, 0], centroids[:, 1], c='red', s=200, marker='X')
    axes[0].set_title(f"Inicializace (Inertia: {kmeans.inertia_:.2f})")
    axes[0].grid(True)
    
    # Postupná iterace a vizualizace
    for i in range(1, max_iters):
        # Aktualizace centroidů
        kmeans = KMeans(n_clusters=n_clusters, random_state=0, max_iter=i+1, n_init=1, init=centroids)
        kmeans.fit(X)
        centroids = kmeans.cluster_centers_
        labels = kmeans.labels_
        
        # Vizualizace aktuálního stavu
        axes[i].scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=50, alpha=0.7)
        axes[i].scatter(centroids[:, 0], centroids[:, 1], c='red', s=200, marker='X')
        axes[i].set_title(f"Iterace {i} (Inertia: {kmeans.inertia_:.2f})")
        axes[i].grid(True)
    
    plt.tight_layout()
    plt.show()

# Vizualizace procesu KMeans
kmeans_step_by_step(X)

## 2. Určení optimálního počtu shluků

Jedním z hlavních problémů K-Means je nutnost předem specifikovat počet shluků. Existuje několik metod, jak určit optimální počet shluků:

In [ ]:
def plot_elbow_method(X, max_clusters=10):
    # Výpočet inertia pro různé hodnoty K
    inertia_values = []
    silhouette_scores = []
    ch_scores = []
    db_scores = []
    
    for k in range(1, max_clusters + 1):
        if k == 1:
            # Pro K=1 nelze vypočítat silhouette score
            kmeans = KMeans(n_clusters=k, random_state=0)
            kmeans.fit(X)
            inertia_values.append(kmeans.inertia_)
            silhouette_scores.append(0)  # Placeholder
            ch_scores.append(0)  # Placeholder
            db_scores.append(0)  # Placeholder
        else:
            kmeans = KMeans(n_clusters=k, random_state=0)
            labels = kmeans.fit_predict(X)
            inertia_values.append(kmeans.inertia_)
            silhouette_scores.append(silhouette_score(X, labels))
            ch_scores.append(calinski_harabasz_score(X, labels))
            db_scores.append(davies_bouldin_score(X, labels))
    
    # Vytvoření grafů
    fig, axs = plt.subplots(2, 2, figsize=(16, 12))
    
    # Elbow Method
    axs[0, 0].plot(range(1, max_clusters + 1), inertia_values, marker='o')
    axs[0, 0].set_title('Elbow Method', fontsize=14)
    axs[0, 0].set_xlabel('Počet shluků', fontsize=12)
    axs[0, 0].set_ylabel('Inertia', fontsize=12)
    axs[0, 0].grid(True)
    
    # Silhouette Score
    axs[0, 1].plot(range(1, max_clusters + 1), silhouette_scores, marker='o')
    axs[0, 1].set_title('Silhouette Score', fontsize=14)
    axs[0, 1].set_xlabel('Počet shluků', fontsize=12)
    axs[0, 1].set_ylabel('Silhouette Score', fontsize=12)
    axs[0, 1].grid(True)
    
    # Calinski-Harabasz Index
    axs[1, 0].plot(range(1, max_clusters + 1), ch_scores, marker='o')
    axs[1, 0].set_title('Calinski-Harabasz Index', fontsize=14)
    axs[1, 0].set_xlabel('Počet shluků', fontsize=12)
    axs[1, 0].set_ylabel('CH Index', fontsize=12)
    axs[1, 0].grid(True)
    
    # Davies-Bouldin Index
    axs[1, 1].plot(range(1, max_clusters + 1), db_scores, marker='o')
    axs[1, 1].set_title('Davies-Bouldin Index', fontsize=14)
    axs[1, 1].set_xlabel('Počet shluků', fontsize=12)
    axs[1, 1].set_ylabel('DB Index', fontsize=12)
    axs[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Výpis optimálního počtu shluků podle různých metrik
    print("Optimální počet shluků podle různých metrik:")
    # Elbow Method - hledáme "ohyb" v křivce
    print(f"  Elbow Method: Vizuální inspekce grafu")
    # Silhouette Score - vyšší je lepší
    best_k_silhouette = np.argmax(silhouette_scores[1:]) + 2  # +1 pro index, +1 pro začátek od K=2
    print(f"  Silhouette Score: {best_k_silhouette} (skóre: {silhouette_scores[best_k_silhouette-1]:.3f})")
    # Calinski-Harabasz Index - vyšší je lepší
    best_k_ch = np.argmax(ch_scores[1:]) + 2
    print(f"  Calinski-Harabasz Index: {best_k_ch} (skóre: {ch_scores[best_k_ch-1]:.3f})")
    # Davies-Bouldin Index - nižší je lepší
    best_k_db = np.argmin(db_scores[1:]) + 2
    print(f"  Davies-Bouldin Index: {best_k_db} (skóre: {db_scores[best_k_db-1]:.3f})")

# Aplikace metod pro určení optimálního počtu shluků
plot_elbow_method(X, max_clusters=8)

## 3. Alternativní inicializační metody

K-Means je citlivý na inicializaci centroidů. Scikit-learn nabízí několik inicializačních strategií:

In [ ]:
# Vytvoření nových syntetických dat se 4 překrývajícími se shluky
X_complex, y_complex = make_blobs(n_samples=500, centers=4, cluster_std=1.2, random_state=42)

# Přidání šumu
X_complex = np.vstack([X_complex, np.random.randn(50, 2) * 4])

# Vizualizace dat
plt.figure(figsize=(10, 6))
plt.scatter(X_complex[:, 0], X_complex[:, 1], alpha=0.7)
plt.title('Komplexnější syntetická data', fontsize=14)
plt.xlabel('První příznak', fontsize=12)
plt.ylabel('Druhý příznak', fontsize=12)
plt.grid(True)
plt.show()

# Porovnání různých inicializačních metod
init_methods = ['k-means++', 'random']
fig, axs = plt.subplots(1, len(init_methods), figsize=(16, 6))

for i, method in enumerate(init_methods):
    # Měření času provedení
    start_time = time.time()
    
    # Aplikace K-Means s danou inicializační metodou
    kmeans = KMeans(n_clusters=4, init=method, random_state=42)
    labels = kmeans.fit_predict(X_complex)
    centroids = kmeans.cluster_centers_
    
    execution_time = time.time() - start_time
    
    # Vizualizace výsledků
    axs[i].scatter(X_complex[:, 0], X_complex[:, 1], c=labels, cmap='viridis', alpha=0.7)
    axs[i].scatter(centroids[:, 0], centroids[:, 1], c='red', s=200, marker='X')
    axs[i].set_title(f"Inicializace: {method}\nInertia: {kmeans.inertia_:.2f}, Iterace: {kmeans.n_iter_}\nČas: {execution_time:.4f}s", fontsize=12)
    axs[i].grid(True)
    
    # Výpočet metrik
    silhouette = silhouette_score(X_complex, labels)
    print(f"Metoda {method}:")
    print(f"  Inertia: {kmeans.inertia_:.2f}")
    print(f"  Počet iterací: {kmeans.n_iter_}")
    print(f"  Silhouette skóre: {silhouette:.3f}")
    print(f"  Čas provedení: {execution_time:.4f}s\n")

plt.tight_layout()
plt.show()

## 4. K-Means na různých tvarech dat

K-Means předpokládá, že shluky mají sférický (kulovitý) tvar a jsou přibližně stejné velikosti. Podívejme se, jak si K-Means poradí s daty, která porušují tento předpoklad.

In [ ]:
# Vytvoření různých tvarů dat
plt.figure(figsize=(18, 6))

# 1. Sférické shluky (ideální pro K-Means)
X_spherical, y_spherical = make_blobs(n_samples=500, centers=3, cluster_std=0.7, random_state=0)

# 2. Shluky s různou velikostí a hustotou
X_varied = np.vstack([
    np.random.randn(100, 2) * 0.3 + [-2, -2],  # Malý, hustý shluk
    np.random.randn(50, 2) * 1.5 + [1, -2],    # Velký, řídký shluk
    np.random.randn(300, 2) * 0.8 + [0, 2]     # Střední shluk
])
y_varied = np.hstack([np.zeros(100), np.ones(50), np.ones(300) * 2])

# 3. Nelineární shluky (dva půlměsíce)
X_moons, y_moons = make_moons(n_samples=500, noise=0.1, random_state=42)

# Typy dat pro testování
datasets = {
    'Sférické shluky': (X_spherical, y_spherical, 3),
    'Shluky s různou velikostí': (X_varied, y_varied, 3),
    'Nelineární shluky': (X_moons, y_moons, 2)
}

# Vizualizace dat a K-Means výsledků
fig, axs = plt.subplots(2, 3, figsize=(18, 12))

for i, (name, (X, y, n_clusters)) in enumerate(datasets.items()):
    # Zobrazení původních dat
    axs[0, i].scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', alpha=0.7)
    axs[0, i].set_title(f"{name} - Skutečné třídy", fontsize=14)
    axs[0, i].grid(True)
    
    # Aplikace K-Means
    kmeans = KMeans(n_clusters=n_clusters, random_state=0)
    labels = kmeans.fit_predict(X)
    centroids = kmeans.cluster_centers_
    
    # Zobrazení výsledků K-Means
    axs[1, i].scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', alpha=0.7)
    axs[1, i].scatter(centroids[:, 0], centroids[:, 1], c='red', s=200, marker='X')
    
    # Výpočet metrik
    silhouette = silhouette_score(X, labels)
    ch_score = calinski_harabasz_score(X, labels)
    axs[1, i].set_title(f"K-Means výsledky\nSilhouette: {silhouette:.3f}, CH: {ch_score:.1f}", fontsize=14)
    axs[1, i].grid(True)

plt.tight_layout()
plt.show()

## 5. K-Means na reálných datech

Nyní aplikujeme K-Means na reálná data - iris dataset, který obsahuje měření různých druhů kosatců.

In [ ]:
# Načtení iris datasetu
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Základní informace o datasetu
print(f"Iris dataset:")
print(f"  Počet vzorků: {X_iris.shape[0]}")
print(f"  Počet příznaků: {X_iris.shape[1]}")
print(f"  Názvy příznaků: {iris.feature_names}")
print(f"  Třídy: {iris.target_names}")
print(f"  Počet instancí v každé třídě: {np.bincount(y_iris)}")

# Aplikace K-Means
kmeans_iris = KMeans(n_clusters=3, random_state=0)
labels_iris = kmeans_iris.fit_predict(X_iris)

# Vizualizace výsledků - použijeme PCA pro redukci dimenzionality do 2D
pca = PCA(n_components=2)
X_iris_pca = pca.fit_transform(X_iris)

plt.figure(figsize=(16, 6))

# Původní třídy
plt.subplot(1, 2, 1)
plt.scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], c=y_iris, cmap='viridis', alpha=0.7)
plt.title('Iris dataset - skutečné třídy', fontsize=14)
plt.xlabel('První hlavní komponenta', fontsize=12)
plt.ylabel('Druhá hlavní komponenta', fontsize=12)
plt.colorbar(label='Třída')
plt.grid(True)

# K-Means výsledky
plt.subplot(1, 2, 2)
plt.scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], c=labels_iris, cmap='viridis', alpha=0.7)
pca_centroids = pca.transform(kmeans_iris.cluster_centers_)
plt.scatter(pca_centroids[:, 0], pca_centroids[:, 1], c='red', s=200, marker='X')
plt.title('K-Means shluky (K=3)', fontsize=14)
plt.xlabel('První hlavní komponenta', fontsize=12)
plt.ylabel('Druhá hlavní komponenta', fontsize=12)
plt.colorbar(label='Shluk')
plt.grid(True)

plt.tight_layout()
plt.show()

# Hodnocení výsledků K-Means
silhouette_iris = silhouette_score(X_iris, labels_iris)
ch_iris = calinski_harabasz_score(X_iris, labels_iris)
db_iris = davies_bouldin_score(X_iris, labels_iris)

print("\nMetriky kvality shlukování na Iris datasetu:")
print(f"  Silhouette koeficient: {silhouette_iris:.3f} (vyšší je lepší)")
print(f"  Calinski-Harabasz index: {ch_iris:.3f} (vyšší je lepší)")
print(f"  Davies-Bouldin index: {db_iris:.3f} (nižší je lepší)")

In [ ]:
# Analýza shody mezi skutečnými třídami a K-Means shluky
from sklearn.metrics import confusion_matrix, adjusted_rand_score, normalized_mutual_info_score

# Kontingenční tabulka (kolik instancí v každé třídě patří do kterého shluku)
contingency = pd.crosstab(y_iris, labels_iris, rownames=['Skutečná třída'], colnames=['Shluk'])
print("Kontingenční tabulka:")
print(contingency)

# Metriky pro porovnání skutečných tříd a predikovaných shluků
ari = adjusted_rand_score(y_iris, labels_iris)
nmi = normalized_mutual_info_score(y_iris, labels_iris)

print(f"\nAdjusted Rand Index: {ari:.3f} (rozsah: -1 až 1, vyšší je lepší)")
print(f"Normalized Mutual Information: {nmi:.3f} (rozsah: 0 až 1, vyšší je lepší)")

# Analýza charakteristik jednotlivých shluků
print("\nCharakteristiky shluků:")
iris_df = pd.DataFrame(X_iris, columns=iris.feature_names)
iris_df['shluk'] = labels_iris

# Průměrné hodnoty příznaků pro každý shluk
cluster_means = iris_df.groupby('shluk').mean()
print(cluster_means)

## 6. Vizualizace vlivu škálování na K-Means

K-Means používá Euklidovskou vzdálenost, proto je citlivý na škálování příznaků. Ukážeme si, jak škálování dat může ovlivnit výsledky.

In [ ]:
# Vytvoření dat s různě škálovanými příznaky
X_scaled_test = np.zeros((300, 2))
X_scaled_test[:100, 0] = np.random.normal(0, 1, 100)  # První shluk
X_scaled_test[:100, 1] = np.random.normal(0, 1, 100)
X_scaled_test[100:200, 0] = np.random.normal(5, 1, 100)  # Druhý shluk
X_scaled_test[100:200, 1] = np.random.normal(5, 1, 100)
X_scaled_test[200:300, 0] = np.random.normal(0, 1, 100)  # Třetí shluk
X_scaled_test[200:300, 1] = np.random.normal(5, 1, 100)

# Škálování druhého příznaku
X_scaled_test_mod = X_scaled_test.copy()
X_scaled_test_mod[:, 1] = X_scaled_test_mod[:, 1] * 10  # Zvětšení měřítka 10x

# Definujeme pravdivé shluky pro srovnání
true_labels = np.concatenate([np.zeros(100), np.ones(100), np.ones(100) * 2])

# K-Means na původních a škálovaných datech
fig, axs = plt.subplots(2, 2, figsize=(16, 12))

# Původní data
axs[0, 0].scatter(X_scaled_test[:, 0], X_scaled_test[:, 1], c=true_labels, cmap='viridis', alpha=0.7)
axs[0, 0].set_title('Původní data - skutečné shluky', fontsize=14)
axs[0, 0].grid(True)

# K-Means na původních datech
kmeans_orig = KMeans(n_clusters=3, random_state=0)
labels_orig = kmeans_orig.fit_predict(X_scaled_test)
axs[0, 1].scatter(X_scaled_test[:, 0], X_scaled_test[:, 1], c=labels_orig, cmap='viridis', alpha=0.7)
axs[0, 1].scatter(kmeans_orig.cluster_centers_[:, 0], kmeans_orig.cluster_centers_[:, 1], 
                c='red', s=200, marker='X')
axs[0, 1].set_title(f'K-Means na původních datech\nARI: {adjusted_rand_score(true_labels, labels_orig):.3f}', fontsize=14)
axs[0, 1].grid(True)

# Škálovaná data
axs[1, 0].scatter(X_scaled_test_mod[:, 0], X_scaled_test_mod[:, 1], c=true_labels, cmap='viridis', alpha=0.7)
axs[1, 0].set_title('Škálovaná data (2. příznak * 10) - skutečné shluky', fontsize=14)
axs[1, 0].grid(True)

# K-Means na škálovaných datech
kmeans_scaled = KMeans(n_clusters=3, random_state=0)
labels_scaled = kmeans_scaled.fit_predict(X_scaled_test_mod)
axs[1, 1].scatter(X_scaled_test_mod[:, 0], X_scaled_test_mod[:, 1], c=labels_scaled, cmap='viridis', alpha=0.7)
axs[1, 1].scatter(kmeans_scaled.cluster_centers_[:, 0], kmeans_scaled.cluster_centers_[:, 1], 
                 c='red', s=200, marker='X')
axs[1, 1].set_title(f'K-Means na škálovaných datech\nARI: {adjusted_rand_score(true_labels, labels_scaled):.3f}', fontsize=14)
axs[1, 1].grid(True)

plt.tight_layout()
plt.show()

# Demonstrace efektu škálování a standardizace
plt.figure(figsize=(16, 6))

# Před standardizací
plt.subplot(1, 2, 1)
kmeans_raw = KMeans(n_clusters=3, random_state=0)
labels_raw = kmeans_raw.fit_predict(X_scaled_test_mod)
plt.scatter(X_scaled_test_mod[:, 0], X_scaled_test_mod[:, 1], c=labels_raw, cmap='viridis', alpha=0.7)
plt.scatter(kmeans_raw.cluster_centers_[:, 0], kmeans_raw.cluster_centers_[:, 1], 
           c='red', s=200, marker='X')
plt.title('K-Means na neškálovaných datech', fontsize=14)
plt.grid(True)

# Po standardizaci
plt.subplot(1, 2, 2)
scaler = StandardScaler()
X_standardized = scaler.fit_transform(X_scaled_test_mod)
kmeans_std = KMeans(n_clusters=3, random_state=0)
labels_std = kmeans_std.fit_predict(X_standardized)
plt.scatter(X_standardized[:, 0], X_standardized[:, 1], c=labels_std, cmap='viridis', alpha=0.7)
plt.scatter(kmeans_std.cluster_centers_[:, 0], kmeans_std.cluster_centers_[:, 1], 
           c='red', s=200, marker='X')
plt.title(f'K-Means na standardizovaných datech\nARI: {adjusted_rand_score(true_labels, labels_std):.3f}', fontsize=14)
plt.grid(True)

plt.tight_layout()
plt.show()

## 7. MiniBatch K-Means pro velké datové sady

Pro velmi velké datové sady můžeme použít MiniBatch K-Means, což je optimalizovaná verze K-Means, která zpracovává data po malých dávkách.

In [ ]:
from sklearn.cluster import MiniBatchKMeans
import time

# Vytvoření většího datasetu pro demonstraci
X_large, y_large = make_blobs(n_samples=10000, centers=5, random_state=42)

# Porovnání standardního K-Means a MiniBatch K-Means
algorithms = {
    'K-Means': KMeans(n_clusters=5, random_state=0),
    'MiniBatch K-Means': MiniBatchKMeans(n_clusters=5, batch_size=100, random_state=0)
}

results = {}
fig, axs = plt.subplots(1, 2, figsize=(16, 6))

for i, (name, algorithm) in enumerate(algorithms.items()):
    # Měření času
    start_time = time.time()
    labels = algorithm.fit_predict(X_large)
    execution_time = time.time() - start_time
    
    # Výpočet metrik
    silhouette = silhouette_score(X_large, labels)
    inertia = algorithm.inertia_
    
    # Uložení výsledků
    results[name] = {
        'time': execution_time,
        'silhouette': silhouette,
        'inertia': inertia
    }
    
    # Vizualizace výsledků
    sc = axs[i].scatter(X_large[:, 0], X_large[:, 1], c=labels, cmap='viridis', alpha=0.5, s=10)
    axs[i].scatter(algorithm.cluster_centers_[:, 0], algorithm.cluster_centers_[:, 1], 
                  c='red', s=200, marker='X')
    axs[i].set_title(f'{name}\nČas: {execution_time:.3f}s, Silhouette: {silhouette:.3f}\nInertia: {inertia:.1f}', fontsize=14)
    axs[i].grid(True)
    plt.colorbar(sc, ax=axs[i], label='Cluster')

plt.tight_layout()
plt.show()

# Výpis výsledků
print("Porovnání K-Means a MiniBatch K-Means:")
for name, metrics in results.items():
    print(f"\n{name}:")
    print(f"  Čas provedení: {metrics['time']:.4f} sekund")
    print(f"  Silhouette skóre: {metrics['silhouette']:.4f}")
    print(f"  Inertia: {metrics['inertia']:.2f}")

# Poměr zrychlení
speedup = results['K-Means']['time'] / results['MiniBatch K-Means']['time']
print(f"\nMiniBatch K-Means je {speedup:.2f}x rychlejší než standardní K-Means")

## 8. Shrnutí a doporučení pro použití K-Means

K-Means je jednoduchý, ale efektivní shlukovací algoritmus s několika důležitými vlastnostmi a omezeními. Zde jsou klíčová doporučení:

### Výhody K-Means:
1. **Jednoduchost** - Snadno pochopitelný a implementovatelný algoritmus
2. **Škálovatelnost** - Efektivní na velkých datových sadách (zvláště MiniBatch verze)
3. **Flexibilita** - Lze použít pro různé typy dat (po vhodném předzpracování)
4. **Rychlost** - Rychlá konvergence a lineární složitost vzhledem k počtu vzorků

### Nevýhody K-Means:
1. **Nutnost předem specifikovat K** - Potřeba znát nebo odhadnout počet shluků
2. **Citlivost na inicializaci** - Výsledky se mohou lišit v závislosti na počáteční volbě centroidů
3. **Předpoklad kulovitých shluků** - Problémy s nelineárními, komplexními strukturami
4. **Citlivost na odlehlé hodnoty** - Odlehlé hodnoty mohou výrazně ovlivnit výsledky
5. **Citlivost na škálování příznaků** - Příznaky s větším rozsahem hodnot budou mít větší váhu

### Klíčová doporučení při používání K-Means:

1. **Předzpracování dat**:
   - Standardizujte nebo normalizujte data před použitím K-Means
   - Zvažte odstranění nebo zpracování odlehlých hodnot
   - Zvažte redukci dimenzionality pro vysoko-dimenzionální data (např. pomocí PCA)

2. **Volba počtu shluků (K)**:
   - Použijte Elbow Method, Silhouette Score nebo další metriky pro odhad optimálního K
   - Zvažte doménovou znalost problému (pokud máte předem znalosti o pravděpodobném počtu shluků)
   - V některých případech může být vhodné vyzkoušet několik různých hodnot K a porovnat výsledky

3. **Vylepšení stability výsledků**:
   - Používejte inicializaci 'k-means++' místo náhodné inicializace
   - Nastavte větší počet inicializací (parametr `n_init`)
   - Nastavte pevný `random_state` pro reprodukovatelnost výsledků

4. **Pro velké datové sady**:
   - Použijte MiniBatch K-Means pro efektivnější zpracování
   - Optimalizujte velikost dávky (`batch_size`) pro vyvážení rychlosti a přesnosti

5. **Interpretace výsledků**:
   - Analyzujte vlastnosti shluků (centroidy, rozptyl, velikost)
   - Vizualizujte data a shluky, pokud je to možné
   - Vyhodnoťte smysluplnost shluků z hlediska doménové znalosti

6. **Když K-Means selhává**:
   - Pro nelineární struktury zvažte algoritmy jako DBSCAN, Spectral Clustering nebo Gaussian Mixture Models
   - Pro shluky s různou hustotou zvažte DBSCAN nebo HDBSCAN
   - Pro shluky různých velikostí zvažte Gaussian Mixture Models
   
### Typické aplikace K-Means:
- Segmentace zákazníků v marketingu
- Komprese obrazu
- Analýza dokumentů a témat
- Detekce anomálií (jako předzpracování)
- Zjednodušení dat pro další analýzu

K-Means je často dobrým výchozím bodem pro shlukovací analýzu díky své jednoduchosti a efektivitě, ale je důležité znát jeho omezení a vědět, kdy je vhodné použít pokročilejší metody.